In [5]:
"""
=============================================================
TELECOM ALARM DATASET — EXPLORATORY DATA ANALYSIS (EDA)
=============================================================
Sections:
  1. Basic Dataset Overview
  2. Distribution Analysis (Inactive Time & Fluctuation)
  3. Hourly Pattern Analysis
  4. Day-of-Week Pattern Analysis
  5. Vendor-wise Behavior Analysis
  6. Region-wise Behavior Analysis
  7. Correlation Analysis
  8. Threshold Label Visualization
  9. EDA Summary CSV

All plots saved to: ./eda_outputs/
Summary saved to:   ./eda_outputs/eda_summary.csv
=============================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# ── Setup ─────────────────────────────────────────────────────
OUTPUT_DIR = './eda_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Publication-quality plot settings
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 14,
})

PALETTE = sns.color_palette("Set2", 10)
sns.set_style("whitegrid")

# ── Load Data ─────────────────────────────────────────────────
print("Loading anonymized dataset...")
df = pd.read_csv('anonymized_alarm_data.csv')
df['Alarm Raised Date'] = pd.to_datetime(df['Alarm Raised Date'])
df['DayOfWeek'] = df['Alarm Raised Date'].dt.day_name()
df['DayOfWeekNum'] = df['Alarm Raised Date'].dt.dayofweek
df['Month'] = df['Alarm Raised Date'].dt.month
def extract_region(cell):
    if cell.startswith('CELL_'):
        return 'REG_OTHER'
    parts = cell.split('_')
    return '_'.join(parts[:2]) if len(parts) >= 2 else cell

df['Region'] = df['Cell'].apply(extract_region)

inactive_cols = [f'Hour_{h}_Inactive' for h in range(1, 25)]
fluct_cols = [f'Hour_{h}_Fluctuation' for h in range(1, 25)]
hours = list(range(1, 25))

print(f"  Shape: {df.shape}")
print(f"  Dates: {df['Alarm Raised Date'].min().date()} to {df['Alarm Raised Date'].max().date()}")
print(f"  Vendors: {df['Vendor'].unique().tolist()}")
print(f"  Regions: {df['Region'].unique().tolist()}")

summary_stats = {}

# =============================================================
# SECTION 1 — BASIC DATASET OVERVIEW
# =============================================================
print("\n[1/8] Basic Dataset Overview...")

overview = {
    'Total Records': len(df),
    'Unique Cells': df['Cell'].nunique(),
    'Unique Dates': df['Alarm Raised Date'].nunique(),
    'Vendors': df['Vendor'].nunique(),
    'Regions': df['Region'].nunique(),
    'Date Range Start': str(df['Alarm Raised Date'].min().date()),
    'Date Range End': str(df['Alarm Raised Date'].max().date()),
    'Null Values': df.isnull().sum().sum(),
}
for k, v in overview.items():
    print(f"  {k}: {v}")

# Vendor distribution pie chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Dataset Overview', fontweight='bold')

vendor_counts = df['Vendor'].value_counts()
axes[0].pie(vendor_counts.values, labels=vendor_counts.index,
            autopct='%1.1f%%', colors=PALETTE[:len(vendor_counts)],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[0].set_title('Vendor Distribution')

region_counts = df['Region'].value_counts()
axes[1].bar(region_counts.index, region_counts.values,
            color=PALETTE[:len(region_counts)], edgecolor='white')
axes[1].set_title('Records per Region')
axes[1].set_xlabel('Region')
axes[1].set_ylabel('Number of Records')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/01_dataset_overview.png')
plt.close()
print("  Saved: 01_dataset_overview.png")

# =============================================================
# SECTION 2 — DISTRIBUTION ANALYSIS
# =============================================================
print("\n[2/8] Distribution Analysis...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribution Analysis — Inactive Time & Fluctuation', fontweight='bold')

# 2a. Total Inactive Time distribution (minutes)
inactive_min = df['Total_Inactive_Time_Seconds'] / 60
axes[0, 0].hist(inactive_min[inactive_min <= 120], bins=60,
                color=PALETTE[0], edgecolor='white', alpha=0.85)
axes[0, 0].set_title('Total Inactive Time Distribution (≤120 min)')
axes[0, 0].set_xlabel('Inactive Time (minutes)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(inactive_min.median(), color='red', linestyle='--',
                    linewidth=1.5, label=f'Median: {inactive_min.median():.1f} min')
axes[0, 0].axvline(inactive_min.quantile(0.75), color='orange', linestyle='--',
                    linewidth=1.5, label=f'P75: {inactive_min.quantile(0.75):.1f} min')
axes[0, 0].legend()

# 2b. Log-scale inactive time (full range)
axes[0, 1].hist(np.log1p(df['Total_Inactive_Time_Seconds']), bins=60,
                color=PALETTE[1], edgecolor='white', alpha=0.85)
axes[0, 1].set_title('Total Inactive Time (Log Scale — Full Range)')
axes[0, 1].set_xlabel('log(1 + Inactive Time in seconds)')
axes[0, 1].set_ylabel('Frequency')

# 2c. Fluctuation count distribution
fluct_counts = df['Fluctuation_Count'].value_counts().sort_index()
fluct_counts = fluct_counts[fluct_counts.index <= 15]
axes[1, 0].bar(fluct_counts.index, fluct_counts.values,
               color=PALETTE[2], edgecolor='white')
axes[1, 0].set_title('Fluctuation Count Distribution (≤15)')
axes[1, 0].set_xlabel('Fluctuation Count')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_xticks(fluct_counts.index)

# 2d. Box plots by vendor
vendors = df['Vendor'].unique()
data_by_vendor = [df[df['Vendor'] == v]['Total_Inactive_Time_Seconds'].values / 60
                  for v in vendors]
bp = axes[1, 1].boxplot(data_by_vendor, labels=vendors, patch_artist=True,
                         showfliers=False)
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[1, 1].set_title('Inactive Time by Vendor (No Outliers)')
axes[1, 1].set_xlabel('Vendor')
axes[1, 1].set_ylabel('Inactive Time (minutes)')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_distribution_analysis.png')
plt.close()
print("  Saved: 02_distribution_analysis.png")

# Percentile table
percentiles = [25, 50, 60, 70, 75, 80, 85, 90, 95, 99]
pct_inactive = [np.percentile(df['Total_Inactive_Time_Seconds'] / 60, p) for p in percentiles]
pct_fluct = [np.percentile(df['Fluctuation_Count'], p) for p in percentiles]
summary_stats['percentiles'] = pd.DataFrame({
    'Percentile': percentiles,
    'Inactive_Time_Minutes': np.round(pct_inactive, 2),
    'Fluctuation_Count': np.round(pct_fluct, 2)
})
print("  Percentile summary computed.")

# =============================================================
# SECTION 3 — HOURLY PATTERN ANALYSIS
# =============================================================
print("\n[3/8] Hourly Pattern Analysis...")

hourly_inactive_mean = df[inactive_cols].mean()
hourly_inactive_median = df[inactive_cols].median()
hourly_inactive_p75 = df[inactive_cols].quantile(0.75)
hourly_fluct_mean = df[fluct_cols].mean()
hourly_fluct_p90 = df[fluct_cols].quantile(0.90)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Hourly Pattern Analysis', fontweight='bold')

# 3a. Mean hourly inactive time
axes[0, 0].fill_between(hours, hourly_inactive_mean / 60, alpha=0.3, color=PALETTE[0])
axes[0, 0].plot(hours, hourly_inactive_mean / 60, 'o-', color=PALETTE[0],
                linewidth=2, markersize=5)
axes[0, 0].set_title('Mean Inactive Time per Hour')
axes[0, 0].set_xlabel('Hour of Day')
axes[0, 0].set_ylabel('Mean Inactive Time (minutes)')
axes[0, 0].set_xticks(hours)
axes[0, 0].set_xticklabels([f'{h:02d}:00' for h in range(24)], rotation=45, ha='right')

# 3b. Mean vs Median vs P75 inactive
axes[0, 1].plot(hours, hourly_inactive_mean / 60, 'o-', label='Mean', linewidth=2)
axes[0, 1].plot(hours, hourly_inactive_median / 60, 's--', label='Median', linewidth=2)
axes[0, 1].plot(hours, hourly_inactive_p75 / 60, '^:', label='P75', linewidth=2)
axes[0, 1].set_title('Inactive Time — Mean vs Median vs P75')
axes[0, 1].set_xlabel('Hour of Day')
axes[0, 1].set_ylabel('Inactive Time (minutes)')
axes[0, 1].set_xticks(hours)
axes[0, 1].set_xticklabels([f'{h:02d}:00' for h in range(24)], rotation=45, ha='right')
axes[0, 1].legend()

# 3c. Mean hourly fluctuation
axes[1, 0].fill_between(hours, hourly_fluct_mean, alpha=0.3, color=PALETTE[2])
axes[1, 0].plot(hours, hourly_fluct_mean, 'o-', color=PALETTE[2],
                linewidth=2, markersize=5)
axes[1, 0].set_title('Mean Fluctuation Count per Hour')
axes[1, 0].set_xlabel('Hour of Day')
axes[1, 0].set_ylabel('Mean Fluctuation Count')
axes[1, 0].set_xticks(hours)
axes[1, 0].set_xticklabels([f'{h:02d}:00' for h in range(24)], rotation=45, ha='right')

# 3d. Heatmap — hourly inactive time by date
pivot_inactive = pd.DataFrame()
for date in df['Alarm Raised Date'].unique():
    day_df = df[df['Alarm Raised Date'] == date]
    row = day_df[inactive_cols].mean() / 60
    row.name = str(date.date())
    pivot_inactive = pd.concat([pivot_inactive, row.to_frame().T])

pivot_inactive.columns = [f'H{h}' for h in range(1, 25)]
sns.heatmap(pivot_inactive.astype(float), ax=axes[1, 1], cmap='YlOrRd',
            linewidths=0.5, annot=False, fmt='.0f',
            cbar_kws={'label': 'Avg Inactive (min)'})
axes[1, 1].set_title('Hourly Inactive Time Heatmap (by Date)')
axes[1, 1].set_xlabel('Hour of Day')
axes[1, 1].set_ylabel('Date')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/03_hourly_patterns.png')
plt.close()
print("  Saved: 03_hourly_patterns.png")

# Heatmap — hourly fluctuation by date
fig, ax = plt.subplots(figsize=(16, 5))
pivot_fluct = pd.DataFrame()
for date in df['Alarm Raised Date'].unique():
    day_df = df[df['Alarm Raised Date'] == date]
    row = day_df[fluct_cols].mean()
    row.name = str(date.date())
    pivot_fluct = pd.concat([pivot_fluct, row.to_frame().T])

pivot_fluct.columns = [f'H{h}' for h in range(1, 25)]
sns.heatmap(pivot_fluct.astype(float), ax=ax, cmap='Blues',
            linewidths=0.5, annot=True, fmt='.2f',
            cbar_kws={'label': 'Avg Fluctuation Count'})
ax.set_title('Hourly Fluctuation Count Heatmap (by Date)', fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Date')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/03b_hourly_fluctuation_heatmap.png')
plt.close()
print("  Saved: 03b_hourly_fluctuation_heatmap.png")

# =============================================================
# SECTION 4 — DAY OF WEEK PATTERN ANALYSIS
# =============================================================
print("\n[4/8] Day-of-Week Pattern Analysis...")

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_order = [d for d in day_order if d in df['DayOfWeek'].unique()]

day_inactive = df.groupby('DayOfWeek')['Total_Inactive_Time_Seconds'].agg(
    ['mean', 'median', lambda x: np.percentile(x, 75)]).reindex(day_order)
day_inactive.columns = ['Mean', 'Median', 'P75']

day_fluct = df.groupby('DayOfWeek')['Fluctuation_Count'].agg(
    ['mean', 'median', lambda x: np.percentile(x, 90)]).reindex(day_order)
day_fluct.columns = ['Mean', 'Median', 'P90']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Day-of-Week Pattern Analysis', fontweight='bold')

x = np.arange(len(day_order))
width = 0.28
axes[0].bar(x - width, day_inactive['Mean'] / 60, width, label='Mean', color=PALETTE[0], alpha=0.85)
axes[0].bar(x, day_inactive['Median'] / 60, width, label='Median', color=PALETTE[1], alpha=0.85)
axes[0].bar(x + width, day_inactive['P75'] / 60, width, label='P75', color=PALETTE[2], alpha=0.85)
axes[0].set_title('Inactive Time by Day of Week')
axes[0].set_xticks(x)
axes[0].set_xticklabels(day_order, rotation=30, ha='right')
axes[0].set_ylabel('Inactive Time (minutes)')
axes[0].legend()

axes[1].bar(x - width, day_fluct['Mean'], width, label='Mean', color=PALETTE[3], alpha=0.85)
axes[1].bar(x, day_fluct['Median'], width, label='Median', color=PALETTE[4], alpha=0.85)
axes[1].bar(x + width, day_fluct['P90'], width, label='P90', color=PALETTE[5], alpha=0.85)
axes[1].set_title('Fluctuation Count by Day of Week')
axes[1].set_xticks(x)
axes[1].set_xticklabels(day_order, rotation=30, ha='right')
axes[1].set_ylabel('Fluctuation Count')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/04_day_of_week_patterns.png')
plt.close()
print("  Saved: 04_day_of_week_patterns.png")

summary_stats['day_of_week_inactive'] = day_inactive.round(2)
summary_stats['day_of_week_fluct'] = day_fluct.round(2)

# =============================================================
# SECTION 5 — VENDOR-WISE BEHAVIOR ANALYSIS
# =============================================================
print("\n[5/8] Vendor-wise Behavior Analysis...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Vendor-wise Behavior Analysis', fontweight='bold')

# 5a. Vendor inactive time box
vendor_list = df['Vendor'].unique().tolist()
data_v = [df[df['Vendor'] == v]['Total_Inactive_Time_Seconds'].values / 60 for v in vendor_list]
bp = axes[0, 0].boxplot(data_v, labels=vendor_list, patch_artist=True, showfliers=False)
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[0, 0].set_title('Inactive Time Distribution by Vendor')
axes[0, 0].set_ylabel('Inactive Time (minutes)')

# 5b. Vendor fluctuation box
data_vf = [df[df['Vendor'] == v]['Fluctuation_Count'].values for v in vendor_list]
bp2 = axes[0, 1].boxplot(data_vf, labels=vendor_list, patch_artist=True, showfliers=False)
for patch, color in zip(bp2['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[0, 1].set_title('Fluctuation Count Distribution by Vendor')
axes[0, 1].set_ylabel('Fluctuation Count')

# 5c. Hourly inactive pattern per vendor
for i, vendor in enumerate(vendor_list):
    vdf = df[df['Vendor'] == vendor]
    mean_hourly = vdf[inactive_cols].mean() / 60
    axes[1, 0].plot(hours, mean_hourly, 'o-', label=vendor,
                    color=PALETTE[i], linewidth=2, markersize=4)
axes[1, 0].set_title('Hourly Inactive Time Pattern by Vendor')
axes[1, 0].set_xlabel('Hour of Day')
axes[1, 0].set_ylabel('Mean Inactive Time (minutes)')
axes[1, 0].set_xticks(hours[::2])
axes[1, 0].set_xticklabels([f'{h:02d}:00' for h in range(0, 24, 2)], rotation=45, ha='right')
axes[1, 0].legend()

# 5d. Hourly fluctuation pattern per vendor
for i, vendor in enumerate(vendor_list):
    vdf = df[df['Vendor'] == vendor]
    mean_hourly = vdf[fluct_cols].mean()
    axes[1, 1].plot(hours, mean_hourly, 'o-', label=vendor,
                    color=PALETTE[i], linewidth=2, markersize=4)
axes[1, 1].set_title('Hourly Fluctuation Pattern by Vendor')
axes[1, 1].set_xlabel('Hour of Day')
axes[1, 1].set_ylabel('Mean Fluctuation Count')
axes[1, 1].set_xticks(hours[::2])
axes[1, 1].set_xticklabels([f'{h:02d}:00' for h in range(0, 24, 2)], rotation=45, ha='right')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/05_vendor_analysis.png')
plt.close()
print("  Saved: 05_vendor_analysis.png")

# =============================================================
# SECTION 6 — REGION-WISE BEHAVIOR ANALYSIS
# =============================================================
print("\n[6/8] Region-wise Behavior Analysis...")

region_list = sorted(df['Region'].unique().tolist())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Region-wise Behavior Analysis', fontweight='bold')

reg_inactive = df.groupby('Region')['Total_Inactive_Time_Seconds'].mean() / 60
reg_inactive = reg_inactive.reindex(region_list)
axes[0].bar(region_list, reg_inactive.values,
            color=PALETTE[:len(region_list)], edgecolor='white')
axes[0].set_title('Mean Inactive Time by Region')
axes[0].set_xlabel('Region')
axes[0].set_ylabel('Mean Inactive Time (minutes)')
axes[0].tick_params(axis='x', rotation=30)

reg_fluct = df.groupby('Region')['Fluctuation_Count'].mean()
reg_fluct = reg_fluct.reindex(region_list)
axes[1].bar(region_list, reg_fluct.values,
            color=PALETTE[:len(region_list)], edgecolor='white')
axes[1].set_title('Mean Fluctuation Count by Region')
axes[1].set_xlabel('Region')
axes[1].set_ylabel('Mean Fluctuation Count')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/06_region_analysis.png')
plt.close()
print("  Saved: 06_region_analysis.png")

# Region × Vendor stacked bar
fig, ax = plt.subplots(figsize=(14, 6))
region_vendor = df.groupby(['Region', 'Vendor']).size().unstack(fill_value=0)
region_vendor.plot(kind='bar', stacked=True, ax=ax,
                   color=PALETTE[:len(vendor_list)], edgecolor='white')
ax.set_title('Record Count by Region and Vendor', fontweight='bold')
ax.set_xlabel('Region')
ax.set_ylabel('Number of Records')
ax.tick_params(axis='x', rotation=30)
ax.legend(title='Vendor')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/06b_region_vendor_breakdown.png')
plt.close()
print("  Saved: 06b_region_vendor_breakdown.png")

# =============================================================
# SECTION 7 — CORRELATION ANALYSIS
# =============================================================
print("\n[7/8] Correlation Analysis...")

# 7a. Correlation between key features
key_features = ['Total_Inactive_Time_Seconds', 'Fluctuation_Count'] + \
               inactive_cols[::4] + fluct_cols[::4]  # every 4th hour to keep readable
corr_df = df[key_features].corr()

fig, ax = plt.subplots(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_df, dtype=bool))
sns.heatmap(corr_df, mask=mask, annot=False, cmap='coolwarm',
            center=0, ax=ax, linewidths=0.3,
            cbar_kws={'label': 'Pearson Correlation'})
ax.set_title('Feature Correlation Matrix (Key Features)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/07_correlation_matrix.png')
plt.close()
print("  Saved: 07_correlation_matrix.png")

# 7b. Inactive time vs Fluctuation scatter
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Inactive Time vs Fluctuation Count', fontweight='bold')

for i, vendor in enumerate(vendor_list):
    vdf = df[df['Vendor'] == vendor]
    sample = vdf.sample(min(500, len(vdf)), random_state=42)
    axes[0].scatter(sample['Fluctuation_Count'],
                    sample['Total_Inactive_Time_Seconds'] / 60,
                    alpha=0.4, label=vendor, color=PALETTE[i], s=20)
axes[0].set_title('All Data (sampled)')
axes[0].set_xlabel('Fluctuation Count')
axes[0].set_ylabel('Total Inactive Time (minutes)')
axes[0].legend()
axes[0].set_ylim(0, 200)
axes[0].set_xlim(0, 20)

# Zoomed view
axes[1].scatter(df['Fluctuation_Count'],
                df['Total_Inactive_Time_Seconds'] / 60,
                alpha=0.15, color=PALETTE[0], s=10)
axes[1].set_title('Full Dataset (all vendors)')
axes[1].set_xlabel('Fluctuation Count')
axes[1].set_ylabel('Total Inactive Time (minutes)')
axes[1].set_ylim(0, 400)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/07b_inactive_vs_fluctuation.png')
plt.close()
print("  Saved: 07b_inactive_vs_fluctuation.png")

# =============================================================
# SECTION 8 — THRESHOLD LABEL VISUALIZATION
# =============================================================
print("\n[8/8] Threshold Label Visualization...")

# For each possible 6-hour window start, compute derived thresholds
# using the strategy: P75 inactive × sensitivity, P90 fluctuation
def sensitivity_factor(start_hour):
    """Evening hours = stricter, night/off-peak = relaxed"""
    if 18 <= start_hour <= 23:
        return 0.5   # evening — customer satisfaction priority
    elif 7 <= start_hour <= 11:
        return 1.0   # morning peak — normal
    else:
        return 1.25  # off-peak — relaxed

window_size = 6
threshold_records = []

for start in range(24):
    hours_in_window = [(start + i) % 24 + 1 for i in range(window_size)]
    w_inactive = df[[f'Hour_{h}_Inactive' for h in hours_in_window]].sum(axis=1)
    w_fluct = df[[f'Hour_{h}_Fluctuation' for h in hours_in_window]].sum(axis=1)

    sf = sensitivity_factor(start)
    thresh_inactive = np.percentile(w_inactive, 75) * sf / 60  # in minutes
    thresh_fluct = max(1, np.percentile(w_fluct, 90))
    thresh_each_hour_fluct = max(1, np.percentile(
        df[[f'Hour_{h}_Fluctuation' for h in hours_in_window]].max(axis=1), 90))

    threshold_records.append({
        'Start_Hour': start + 1,
        'End_Hour': (start + window_size - 1) % 24 + 1,
        'Sensitivity_Factor': sf,
        'Threshold_Inactive_Minutes': round(thresh_inactive, 2),
        'Threshold_Fluctuation': round(thresh_fluct, 1),
        'Threshold_Each_Hour_Fluctuation': round(thresh_each_hour_fluct, 1),
        'Window_Label': f'{start+1:02d}:00–{(start+window_size)%24:02d}:00'
    })

threshold_df = pd.DataFrame(threshold_records)
summary_stats['derived_thresholds_6hr'] = threshold_df

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Derived Threshold Labels — 6-Hour Sliding Window', fontweight='bold')

start_hours = threshold_df['Start_Hour']
colors_sf = [PALETTE[0] if sf == 1.0 else PALETTE[2] if sf == 0.5 else PALETTE[4]
             for sf in threshold_df['Sensitivity_Factor']]

axes[0].bar(start_hours, threshold_df['Threshold_Inactive_Minutes'],
            color=colors_sf, edgecolor='white', alpha=0.85)
axes[0].set_title('Derived Inactive Time Threshold by Window Start Hour')
axes[0].set_xlabel('Window Start Hour')
axes[0].set_ylabel('Threshold Inactive Time (minutes)')
axes[0].set_xticks(start_hours)

# Legend for sensitivity
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=PALETTE[0], label='Normal (×1.0) — Morning Peak'),
    Patch(facecolor=PALETTE[2], label='Strict (×0.5) — Evening'),
    Patch(facecolor=PALETTE[4], label='Relaxed (×1.25) — Off-Peak'),
]
axes[0].legend(handles=legend_elements)

axes[1].plot(start_hours, threshold_df['Threshold_Fluctuation'],
             'o-', color=PALETTE[1], linewidth=2, markersize=6, label='Total Fluctuation')
axes[1].plot(start_hours, threshold_df['Threshold_Each_Hour_Fluctuation'],
             's--', color=PALETTE[3], linewidth=2, markersize=6, label='Per-Hour Fluctuation')
axes[1].set_title('Derived Fluctuation Thresholds by Window Start Hour')
axes[1].set_xlabel('Window Start Hour')
axes[1].set_ylabel('Fluctuation Threshold')
axes[1].set_xticks(start_hours)
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/08_threshold_labels.png')
plt.close()
print("  Saved: 08_threshold_labels.png")

# =============================================================
# SECTION 9 — SAVE EDA SUMMARY CSV
# =============================================================
print("\nSaving EDA summary CSV...")

with pd.ExcelWriter(f'{OUTPUT_DIR}/eda_summary.xlsx', engine='openpyxl') as writer:
    # Overview
    pd.DataFrame([overview]).T.rename(columns={0: 'Value'}).to_excel(
        writer, sheet_name='Overview')
    # Percentiles
    summary_stats['percentiles'].to_excel(
        writer, sheet_name='Percentiles', index=False)
    # Day of week
    summary_stats['day_of_week_inactive'].to_excel(
        writer, sheet_name='DayOfWeek_Inactive')
    summary_stats['day_of_week_fluct'].to_excel(
        writer, sheet_name='DayOfWeek_Fluctuation')
    # Derived thresholds
    summary_stats['derived_thresholds_6hr'].to_excel(
        writer, sheet_name='Derived_Thresholds_6hr', index=False)
    # Hourly means
    pd.DataFrame({
        'Hour': hours,
        'Mean_Inactive_Seconds': [df[inactive_cols[h-1]].mean() for h in hours],
        'Mean_Inactive_Minutes': [df[inactive_cols[h-1]].mean() / 60 for h in hours],
        'P75_Inactive_Minutes': [df[inactive_cols[h-1]].quantile(0.75) / 60 for h in hours],
        'Mean_Fluctuation': [df[fluct_cols[h-1]].mean() for h in hours],
        'P90_Fluctuation': [df[fluct_cols[h-1]].quantile(0.90) for h in hours],
    }).to_excel(writer, sheet_name='Hourly_Stats', index=False)

print(f"  Saved: eda_summary.xlsx")

# =============================================================
# DONE
# =============================================================
print("\n" + "="*60)
print("EDA COMPLETE")
print("="*60)
print(f"All outputs saved to: {OUTPUT_DIR}/")
print("\nFiles generated:")
files = [
    "01_dataset_overview.png",
    "02_distribution_analysis.png",
    "03_hourly_patterns.png",
    "03b_hourly_fluctuation_heatmap.png",
    "04_day_of_week_patterns.png",
    "05_vendor_analysis.png",
    "06_region_analysis.png",
    "06b_region_vendor_breakdown.png",
    "07_correlation_matrix.png",
    "07b_inactive_vs_fluctuation.png",
    "08_threshold_labels.png",
    "eda_summary.xlsx",
]
for f in files:
    print(f"  ✅ {f}")
print("\nNext step: Share these outputs and we proceed to preprocessing for ML.")

Loading anonymized dataset...
  Shape: (14054, 60)
  Dates: 2024-06-08 to 2024-06-24
  Vendors: ['Vendor_X', 'Vendor_Z', 'Vendor_Y']
  Regions: ['REG_A', 'REG_B', 'REG_C', 'REG_OTHER', 'REG_D', 'REG_E', 'REG_F', 'REG_G', 'REG_H']

[1/8] Basic Dataset Overview...
  Total Records: 14054
  Unique Cells: 10648
  Unique Dates: 10
  Vendors: 3
  Regions: 9
  Date Range Start: 2024-06-08
  Date Range End: 2024-06-24
  Null Values: 0
  Saved: 01_dataset_overview.png

[2/8] Distribution Analysis...
  Saved: 02_distribution_analysis.png
  Percentile summary computed.

[3/8] Hourly Pattern Analysis...
  Saved: 03_hourly_patterns.png
  Saved: 03b_hourly_fluctuation_heatmap.png

[4/8] Day-of-Week Pattern Analysis...
  Saved: 04_day_of_week_patterns.png

[5/8] Vendor-wise Behavior Analysis...
  Saved: 05_vendor_analysis.png

[6/8] Region-wise Behavior Analysis...
  Saved: 06_region_analysis.png
  Saved: 06b_region_vendor_breakdown.png

[7/8] Correlation Analysis...
  Saved: 07_correlation_matrix.png

In [4]:
pip install pandas numpy matplotlib seaborn openpyxl

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)

   ------------- -------------------------- 1/3 [openpyxl]
   ------------- -------------------------- 1/3 [openpyxl]
   ------------- -------------------------- 1/3 [openpyxl]
   ------------- -------------------------- 1/3 [openpyxl]
   -------------------------- ------------- 2/3 [seaborn]
   -------------------------- ------------- 2/3 [seaborn]
   -------------------------- ------------- 2/3 [seaborn]
   ---------------------------------------- 3/3 [seaborn]

Note: you may need to restart the kernel to use updated packages.
